### Dataset

In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import os
import sys
import numpy as np
from torchvision.transforms.v2 import RandAugment
import torch
from tqdm.notebook import tqdm
from torch.utils.tensorboard import SummaryWriter

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("../models"))

from dataset import get_data_loaders
from backbone import BackBone
from metrics import l_supcon
from utils import deterministic

In [9]:
SEED = 42

# claude recomienda hacer una función así. Me parece bien
# def set_seed(seed: int):
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)
#     random.seed(seed)

# ahi puse deterministc()

# # Before training
# set_seed(42)
# rand_augment = RandAugment(num_ops=2, magnitude=9)

In [10]:
deterministic(SEED)
dataloaders = get_data_loaders(batch_size=512)

Task [0, 1]: Train=9000, Val=1000, Test=2000
Task [2, 3]: Train=9000, Val=1000, Test=2000
Task [4, 5]: Train=9000, Val=1000, Test=2000
Task [6, 7]: Train=9000, Val=1000, Test=2000
Task [8, 9]: Train=9000, Val=1000, Test=2000


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BackBone().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [12]:
deterministic(SEED)
task0_train = dataloaders[0][0]
rand_augment = RandAugment(num_ops=2, magnitude=9) # Hay que poner seed de vuelta?

tau = 0.3
epochs = 10

writer = SummaryWriter(log_dir="../runs/backbone_training")

for epoch in tqdm(range(epochs), desc = "Epochs", unit = "epoch"):
    batch_bar = tqdm(task0_train, desc = f"Epoch {epoch+1}/{epochs}", leave=False, unit="batch")
    for i, (x, y) in enumerate(batch_bar):
        # x, y = x.to(device), y.to(device)
        # x1 = torch.stack([rand_augment(img) for img in x])
        # x2 = torch.stack([rand_augment(img) for img in x])
        # x_all = torch.cat([x1, x2], dim=0)
        # y_all = torch.cat([y, y], dim=0)
        x_all, y_all = x.to(device), y.to(device)

        optimizer.zero_grad()
        embeddings = model(x_all)
        loss = l_supcon(embeddings, y_all, tau)
        writer.add_scalar("Loss/Train", loss.item(), epoch * len(task0_train) + i)
        print(loss.item())

        loss.backward()
        optimizer.step()

        batch_bar.set_postfix({"loss": loss.item()})
    
writer.close()

Epochs:   0%|          | 0/10 [00:00<?, ?epoch/s]

Epoch 1/10:   0%|          | 0/18 [00:00<?, ?batch/s]

6.2334065437316895
6.242311000823975


KeyboardInterrupt: 

In [ ]:
model.save("../saved_models/backbone.pth")

In [ ]:
np.random.seed(SEED)
# for task_id, (train_loader, val_loader, test_loader) in enumerate (dataloaders):

#     replay_loader = buffer.get_dataloader(batch_size=64) if task_id > 0 else None

#     train(model, train_loader, replay_loader, optimizer, criterion, epochs)

#     validate(model, val_loader)


#     #test contra todas las tareas o solo la ultima?

#     buffer.update(train_loader)